# Task 1 evaluation: results

ACT (chunk 50) trained on **all** 147 demonstrations (`full`) or a **random half** of 73 (`half`), each run with **40** or **15** executed actions per chunk: variants **F40, F15, H40, H15**, evaluated on the same matched configurations in randomised, blind order.

Sections 1–8 are the full analysis. **Section 9 contains only what goes into the report.**

Inputs: `plan/eval_plan.yaml` (pre-registered plan) and, in `A2/results/task1/logs/`, `eval_log.csv` (raw log, never modified), `corrections.yaml` (post-hoc corrections, applied on loading), `pilot_log.csv`. The same results, as a report with all tables and figures, are written by `03_export_results.sh` to `A2/results/task1/README.md`.

In [ ]:
import json
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import analysis as an

pd.set_option("display.max_colwidth", None); pd.set_option("display.width", 200)
plan, log, seqs, applied = an.load()
print("Corrections applied:")
for a in applied:
    print(" -", a)
print("\nAnalysed trials per variant:", {v: len(d) for v, d in seqs.items()}, "| pre-registered n_max:", plan["n_max"])

## 1. Data checks
Rollout counts, invalid (repeated) rollouts, setup accuracy, camera position and control-loop health.

In [ ]:
print("Rows in the raw log:", len(log), "|", log.status.value_counts().to_dict())
print("Every scheduled trial has exactly one valid rollout:",
      sorted(log[log.status == "valid"].trial) == list(range(1, len(pd.read_csv("plan/schedule.csv")) + 1)))
an.data_checks(plan, log, seqs)

In [ ]:
# Invalid rollouts (repeated with the same trial) and excluded trials
log[(log.status == "invalid") | (~log.included & (log.status == "valid"))][
    ["rollout_id", "trial", "block", "config_id", "variant", "status", "end_reason", "duration_s", "setup_override", "notes"]]

In [ ]:
# Rollout recordings present?  Pilot trials (practice, not analysed)
missing = [r for r in log.rollout_id if not (an.RESULTS / "rollouts" / r).is_dir()]
print("Rollouts without a video folder:", missing)
pilot = pd.read_csv(an.RESULTS / "logs" / "pilot_log.csv")
print("Pilot trials:", len(pilot), "|", pilot[["rollout_id", "variant", "success", "end_reason", "duration_s"]].to_dict("records"))

**Scene camera.** Every rollout measured a ~37 px shift of the scene's fixed structure (table edge, robot base) against the demonstrations' median frame (`camera_shift_px`). The camera was in a slightly different position during the evaluation than during data collection. All variants were evaluated under the same shifted view, so the comparison stays fair, but every variant faced a mild distribution shift.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 3))
an_df = pd.concat(seqs.values())
ax[0].hist(an_df.camera_shift_px.clip(upper=80), bins=30); ax[0].set(title="camera shift vs demos (px)", xlabel="px")
ax[1].hist(an_df.cube_err_px.clip(upper=60), bins=30, alpha=0.7, label="cube")
ax[1].hist(an_df.tray_err_px.clip(upper=60), bins=30, alpha=0.7, label="tray")
ax[1].axvline(plan["configurations"]["tolerance_px"], c="k", ls="--"); ax[1].legend()
ax[1].set(title="setup error vs target (px, clipped at 60)", xlabel="px")
ax[2].hist(an_df.limited_cmds, bins=30); ax[2].set(title=f"arm commands limited to {plan['max_step_deg']} deg/step", xlabel="per rollout")
plt.tight_layout()

## 2. Success rate per variant (binary success, 90% Wilson interval)

In [ ]:
summary = an.summary_table(plan, seqs)
summary

## 3. Completion time (successful trials)
From the start of execution until the operator marked the success criterion as met.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
times = [d.loc[d.success == 1, "time_to_success_s"].dropna().astype(float) for d in seqs.values()]
ax.boxplot(times, tick_labels=list(seqs), showfliers=False)
for i, t in enumerate(times, 1):
    ax.scatter(np.random.default_rng(i).normal(i, 0.05, len(t)), t, s=12, alpha=0.7)
ax.set(ylabel="seconds", title="Completion time of successful trials")
plt.tight_layout()

## 4. Task progress and where failures happen
Partial score: 0.25 per completed stage (reach, grasp + lift, carry, place + release). Reported separately from binary success.

In [ ]:
an.failure_stages(seqs)

In [ ]:
an.progress_plot(seqs, path=an.RESULTS / "figures" / "progress.png");

## 5. The two factors (descriptive)
Each factor pooled over the other factor's two levels. Descriptive only: the statistical test is the pairwise STEP comparison in section 7.

In [ ]:
an.main_effects(plan, seqs)

## 6. Per-configuration outcomes
Success (1) or failure (0) of each variant on each configuration, in block order. Shows how much configuration difficulty is shared across variants (the matched design).

In [ ]:
pc = an.per_config(seqs)
fig, ax = plt.subplots(figsize=(4, 9))
ax.imshow(pc[list(seqs)].to_numpy(), cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(len(seqs)), list(seqs)); ax.set_yticks(range(len(pc)), pc.index, fontsize=7)
ax.set_title("success per configuration"); plt.tight_layout()
print("Configurations by number of variants succeeding:", pc["variants succeeding"].value_counts().sort_index().to_dict())

## 7. STEP: all 6 pairs
`MirroredStepTest(inference_mode='ranking', alternative=P0LessThanP1, alpha=0.10/6, n_max=30)`, outcomes fed in trial order, one matched pair at a time; the first decision is recorded.

In [ ]:
pairs = an.step_pairwise(plan, seqs)
rates = {v: d.success.mean() for v, d in seqs.items()}
letters = an.cld_letters(list(seqs), rates, pairs)
print("CLD letters:", letters)
pairs

## 8. Posterior success probability with CLD letters

In [ ]:
an.violin_plot(seqs, letters, path=an.RESULTS / "figures" / "violins.png");

---
# 9. Report results
Everything needed for the one-page write-up. The figures are saved in `A2/results/task1/figures/`.

In [ ]:
import importlib.metadata as md_, subprocess
def version(p):
    try: return md_.version(p)
    except md_.PackageNotFoundError: return "n/a"
k = len(seqs)
n = {v: len(d) for v, d in seqs.items()}
print(f"Design: {k} variants x {min(n.values())} matched configurations = {sum(n.values())} analysed rollouts "
      f"(pre-registered n_max {plan['n_max']}); randomised configuration and variant order; blind labelling.")
print(f"Policies: ACT, chunk 50, 45k steps; full = 147 demos, half = 73 demos. Executed actions per chunk: 40 or 15.")
print(f"Rollout settings: {plan['fps']} Hz, timeout {plan['timeout_s']} s, arm command limit {plan['max_step_deg']} deg/step, "
      f"gripper unlimited, start from the demos' median start pose.")
print(f"STEP: MirroredStepTest, ranking mode, alternative P0LessThanP1, alpha_pair = 0.10/{k*(k-1)//2} = {0.10/(k*(k-1)/2):.4f}, "
      f"n_max {plan['n_max']}, chronological order (no shuffling); sequentialized_barnard_tests {version('sequentialized_barnard_tests')} "
      "(commit 3ae1623).")

### Table 1: results per variant

In [ ]:
t1 = summary.copy()
fs = an.failure_stages(seqs).T
report = pd.DataFrame({
    "policy / exec": [f"{r['policy']} / {r['exec steps']}" for _, r in t1.iterrows()],
    "success": [f"{r['successes']}/{r['trials']}" for _, r in t1.iterrows()],
    "rate [90% CI]": [f"{r['success rate']:.2f} {r['90% Wilson CI']}" for _, r in t1.iterrows()],
    "time median [IQR] s": [f"{r['time median (s)']} {r['time IQR (s)']} (n={r['timed successes']})" for _, r in t1.iterrows()],
    "mean progress": t1["mean progress"].round(2),
    "fail: reach / grasp / carry / release": [" / ".join(str(x) for x in fs.loc[v]) for v in t1.index],
    "timeouts / stopped": [f"{r['timeouts']} / {r['stopped']}" for _, r in t1.iterrows()],
}, index=t1.index)
display(report)
print(an.md_table(report))

### Table 2: STEP pairwise comparisons

In [ ]:
t2 = pairs.assign(**{"result": [f"{r.favoured} better" if r.favoured != "inconclusive" else "inconclusive" for r in pairs.itertuples()],
                     "first decision at trial": pairs["trials per policy at decision"].map(lambda x: "" if pd.isna(x) else int(x))})[
    ["A", "B", "success A", "success B", "decision", "result", "first decision at trial"]]
display(t2)
print(an.md_table(t2, index=False))
print("CLD letters:", letters)

### Figures

In [ ]:
from IPython.display import Image, display
display(Image(an.RESULTS / "figures" / "violins.png")); display(Image(an.RESULTS / "figures" / "progress.png"))

### Factors (descriptive) and deviations from the plan

In [ ]:
display(an.main_effects(plan, seqs))
ex = log[~log.included & (log.status == "valid")]
dev = [
    *applied,
    f"Invalid rollouts (not the policy's fault, repeated): {int((log.status == 'invalid').sum())} "
    f"({', '.join(f'{r.rollout_id}: {r.notes}' for r in log[log.status == 'invalid'].itertuples())}).",
    f"Setup override (started although the detected cube/tray were outside the {plan['configurations']['tolerance_px']} px / "
    f"{plan['configurations']['tolerance_deg']} deg tolerance): {int(an_df.setup_override.sum())} of {len(an_df)} analysed rollouts; "
    f"median setup error cube {an_df.cube_err_px.median():.1f} px, tray {an_df.tray_err_px.median():.1f} px.",
    f"Scene camera shifted by a median {an_df.camera_shift_px.median():.0f} px vs the demonstrations (same for all variants).",
    f"Arm commands were rate-limited ({plan['max_step_deg']} deg/step) in a mean of {an_df.limited_cmds.mean():.0f} steps per rollout.",
    f"Rollouts without a saved video: {missing or 'none'}.",
    f"{len(pilot)} pilot trials on separate configurations (not analysed).",
]
for d in dev:
    print("-", d)

In [ ]:
print("Software:", {p: version(p) for p in ["lerobot", "torch", "sequentialized_barnard_tests", "statistical-comparison-core"]},
      "| repo commit:", subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())